# Chạy thử pipeline RAG – Quy chế đào tạo SGU
Cần Ollama đang chạy (`ollama serve`) và đã pull model trong `src/config.py` (profile `local`: `qwen3:4b`).

In [ ]:
import os, sys, logging
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "eval"))

os.environ["RAG_PROFILE"] = "local"   # đổi thành "server" nếu dùng server Ollama từ xa
logging.getLogger("httpx").setLevel(logging.WARNING)

from config import get_config
cfg = get_config()
print(cfg)

## 0. Khởi động Ollama (bỏ qua nếu đã chạy)
Chỉ dùng được với profile `local`. Profile `server` dùng Ollama từ xa.

In [ ]:
import subprocess, time, urllib.request

def ollama_up() -> bool:
    try:
        urllib.request.urlopen(cfg.ollama_base_url + "/api/tags", timeout=2)
        return True
    except Exception:
        return False

ollama_proc = None
if ollama_up():
    print("Ollama đã chạy sẵn.")
else:
    ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if ollama_up():
            break
        time.sleep(1)
    print("Ollama đã khởi động." if ollama_up() else "Không khởi động được Ollama.")

# Khi xong, tắt tiến trình do notebook mở (chạy ở cell cuối nếu muốn):
# ollama_proc and ollama_proc.terminate()

## 1. Đọc văn bản tầng 2 thành Node (`src/processed_loader.py`)

In [ ]:
from llama_index.core.schema import NodeWithScore
from processed_loader import load_corpus_nodes
from retriever import node_label

nodes = load_corpus_nodes(cfg)   # văn bản tầng 2 theo cfg.corpus_dir / cfg.corpus_files
print(len(nodes), "Node")
for n in nodes[:3]:
    print(node_label(NodeWithScore(node=n)), "|", n.get_content()[:100].replace("\n", " "))

## 2. Build / load Index

In [ ]:
from index_builder import build_or_load_index

index = build_or_load_index(cfg)   # rebuild=True để build lại
print("Số node trong Index:", len(index.docstore.docs))

## 3. Tìm kiếm (`retriever.search`, rerank nếu `cfg.use_rerank`)

In [ ]:
from retriever import search

question = "Điểm trung bình bao nhiêu thì được xếp loại Giỏi?"
for i, n in enumerate(search(index, cfg, question), 1):   # rerank nếu cfg.use_rerank
    print(f"[{i}] score={n.score or 0:.4f} | {node_label(n)} | {n.node.get_content()[:90]!r}")

## 4. Query engine (gọi LLM qua Ollama)

In [ ]:
from query_engine import build_query_engine, ask

engine = build_query_engine(index, cfg)
ask(engine, "Một năm học có bao nhiêu học kỳ?", cfg)

Thử câu hỏi khác:

In [ ]:
ask(engine, "Sinh viên bị buộc thôi học trong những trường hợp nào?", cfg)

## 5. Đánh giá truy hồi theo (văn bản, Điều) — `eval/evaluate.py`

In [ ]:
from evaluate import chay  # pyright: ignore[reportMissingImports]  # eval/ thêm vào sys.path ở cell đầu

ket_qua = chay(lambda q: search(index, cfg, q), (1, 3, 5, 10))   # chấm theo (văn bản, Điều), cả 2 bộ câu hỏi